In [1]:
from typing import Tuple, List, Union, Callable, Any
from contextlib import nullcontext
from itertools import repeat
from collections import UserDict

import torch

from torch import nn, Tensor
from torch.amp import GradScaler, autocast
from torch.utils.checkpoint import get_device_states, set_device_states

class RandContext:
    def __init__(self, *tensors):
        self.fwd_cpu_state = torch.get_rng_state()
        self.fwd_gpu_devices, self.fwd_gpu_states = get_device_states(*tensors)

    def __enter__(self):
        self._fork = torch.random.fork_rng(
            devices=self.fwd_gpu_devices,
            enabled=True
        )
        self._fork.__enter__()
        torch.set_rng_state(self.fwd_cpu_state)
        set_device_states(self.fwd_gpu_devices, self.fwd_gpu_states)

    def __exit__(self, exc_type, exc_val, exc_tb):
        self._fork.__exit__(exc_type, exc_val, exc_tb)
        self._fork = None

class GradCache:
    def __init__(
            self,
            models: List[nn.Module],
            chunk_sizes: Union[int, List[int]],
            loss_fn: Callable[..., Tensor],
            split_input_fn: Callable[[Any, int], Any] = None,
            get_rep_fn: Callable[..., Tensor] = None,
            fp16: bool = False,
            scaler: GradScaler = None,
    ):
        self.models = models

        if isinstance(chunk_sizes, int):
            self.chunk_sizes = [chunk_sizes for _ in range(len(models))]
        else:
            self.chunk_sizes = chunk_sizes

        self.split_input_fn = split_input_fn
        self.get_rep_fn = get_rep_fn
        self.loss_fn = loss_fn

        if fp16:
            assert scaler is not None, "mixed precision training requires a gradient scaler passed in"

        self.fp16 = fp16
        self.scaler = scaler

        self._get_input_tensors_strict = False

    def __call__(self, *args, **kwargs):
        return self.cache_step(*args, **kwargs)

    def split_inputs(self, model_input, chunk_size: int) -> List:
        # delegate splitting to user provided function
        if self.split_input_fn is not None:
            return self.split_input_fn(model_input, chunk_size)

        if isinstance(model_input, (dict, UserDict)) and all(isinstance(x, Tensor) for x in model_input.values()):
            keys = list(model_input.keys())
            chunked_tensors = [model_input[k].split(chunk_size, dim=0) for k in keys]
            return [dict(zip(kk, tt)) for kk, tt in zip(repeat(keys), zip(*chunked_tensors))]

        elif isinstance(model_input, list) and all(isinstance(x, Tensor) for x in model_input):
            chunked_x = [t.split(chunk_size, dim=0) for t in model_input]
            return [list(s) for s in zip(*chunked_x)]

        elif isinstance(model_input, Tensor):
            return list(model_input.split(chunk_size, dim=0))

        elif isinstance(model_input, tuple) and list(map(type, model_input)) == [list, dict]:
            args_chunks = self.split_inputs(model_input[0], chunk_size)
            kwargs_chunks = self.split_inputs(model_input[1], chunk_size)
            return list(zip(args_chunks, kwargs_chunks))

        else:
            raise NotImplementedError(f'Model input split not implemented for type {type(model_input)}')

    def get_input_tensors(self, model_input) -> List[Tensor]:
        if isinstance(model_input, Tensor):
            return [model_input]

        elif isinstance(model_input, (list, tuple)):
            return sum((self.get_input_tensors(x) for x in model_input), [])

        elif isinstance(model_input, (dict, UserDict)):
            return sum((self.get_input_tensors(x) for x in model_input.values()), [])

        elif self._get_input_tensors_strict:
            raise NotImplementedError(f'get_input_tensors not implemented for type {type(model_input)}')

        else:
            return []

    def model_call(self, model: nn.Module, model_input):
        with autocast('cuda') if self.fp16 else nullcontext():
            if isinstance(model_input, Tensor):
                return model(model_input)
            elif isinstance(model_input, list):
                return model(*model_input)
            elif isinstance(model_input, (dict, UserDict)):
                return model(**model_input)
            elif isinstance(model_input, tuple) and list(map(type, model_input)) == [list, dict]:
                model_args, model_kwargs = model_input
                return model(*model_args, **model_kwargs)
            else:
                raise NotImplementedError

    def get_reps(self, model_out) -> Tensor:
        if self.get_rep_fn is not None:
            return self.get_rep_fn(model_out)
        else:
            return model_out

    def compute_loss(self, *reps: Tensor, **loss_kwargs) -> Tensor:
        loss = self.loss_fn(*reps, **loss_kwargs)
        return loss

    def forward_no_grad(
            self,
            model: nn.Module,
            model_inputs,
    ) -> [Tensor, List[RandContext]]:
        rnd_states = []
        model_reps = []

        with torch.no_grad():
            for x in model_inputs:
                rnd_states.append(RandContext(*self.get_input_tensors(x)))
                y = self.model_call(model, x)
                model_reps.append(self.get_reps(y))

        # concatenate all sub-batch representations
        model_reps = torch.cat(model_reps, dim=0)
        return model_reps, rnd_states

    def build_cache(self, *reps: Tensor, **loss_kwargs) -> [List[Tensor], Tensor]:
        reps = [r.detach().requires_grad_() for r in reps]
        with autocast('cuda') if self.fp16 else nullcontext():
            loss = self.compute_loss(*reps, **loss_kwargs)

        if self.fp16:
            self.scaler.scale(loss).backward()
        else:
            loss.backward()

        cache = [r.grad for r in reps]

        return cache, loss.detach()

    def forward_backward(
            self,
            model: nn.Module,
            model_inputs,
            cached_gradients: List[Tensor],
            random_states: List[RandContext],
            no_sync_except_last: bool = False
    ):
        if no_sync_except_last:
            sync_contexts = [model.no_sync for _ in range(len(model_inputs) - 1)] + [nullcontext]
        else:
            sync_contexts = [nullcontext for _ in range(len(model_inputs))]

        for x, state, gradient, sync_context in zip(model_inputs, random_states, cached_gradients, sync_contexts):
            with sync_context():
                with state:
                    y = self.model_call(model, x)
                reps = self.get_reps(y)

                surrogate = torch.dot(reps.flatten(), gradient.flatten())
                surrogate.backward()

    def cache_step(
            self,
            *model_inputs,
            no_sync_except_last: bool = False,
            **loss_kwargs
    ) -> Tensor:
        all_reps = []
        all_rnd_states = []

        if no_sync_except_last:
            assert all(map(lambda m: isinstance(m, nn.parallel.DistributedDataParallel), self.models)), \
                'Some of models are not wrapped in DistributedDataParallel. Make sure you are running DDP with ' \
                'proper initializations.'

        model_inputs = [self.split_inputs(x, chunk_size) for x, chunk_size in zip(model_inputs, self.chunk_sizes)]

        for model, x in zip(self.models, model_inputs):
            model_reps, rnd_states = self.forward_no_grad(model, x)
            all_reps.append(model_reps)
            all_rnd_states.append(rnd_states)

        cache, loss = self.build_cache(*all_reps, **loss_kwargs)
        cache = [c.split(chunk_size) for c, chunk_size in zip(cache, self.chunk_sizes)]

        for model, x, model_cache, rnd_states in zip(
                self.models, model_inputs, cache, all_rnd_states):
            self.forward_backward(model, x, model_cache, rnd_states, no_sync_except_last=no_sync_except_last)

        return loss


class PLGradCache(GradCache):
    def __init__(
        self,
        models: List[nn.Module],
        chunk_sizes: Union[int, List[int]],
        loss_fn: Callable[..., Tensor],
        split_input_fn: Callable[[Any, int], Any] = None,
        get_rep_fn: Callable[..., Tensor] = None,
        fp16: bool = False,
        scaler: GradScaler = None,
        backward_fn=None,  # [added]
    ):
        super().__init__(models, chunk_sizes, loss_fn, split_input_fn, get_rep_fn, fp16, scaler)
        self.backward_fn = backward_fn

    def build_cache(self, *reps: Tensor, **loss_kwargs) -> Union[List[Tensor], Tensor]:
        reps = [r.detach().requires_grad_() for r in reps]
        with autocast('cuda') if self.fp16 else nullcontext():
            loss = self.compute_loss(*reps, **loss_kwargs)

        self.backward_fn(loss)  # [modified]

        cache = [r.grad for r in reps]

        return cache, loss.detach()

    def forward_backward(
        self,
        model: nn.Module,
        model_inputs,
        cached_gradients: List[Tensor],
        random_states: List[RandContext],
        no_sync_except_last: bool = False,
    ):
        if isinstance(
            model, nn.parallel.DistributedDataParallel
        ):  # [use ddp_model]

            if no_sync_except_last:
                sync_contexts = [
                    model.no_sync for _ in range(len(model_inputs) - 1)
                ] + [nullcontext]
                sync_flags = [True] * (len(model_inputs))  # [added]
            else:
                sync_contexts = [nullcontext for _ in range(len(model_inputs))]
                sync_flags = [False] * (len(model_inputs))  # [added]

            # [modified]
            for x, state, gradient, sync_context, sync_flag in zip(
                model_inputs, random_states, cached_gradients, sync_contexts, sync_flags
            ):
                with sync_context():
                    with state:
                        y = self.model_call(model, x)
                    reps = self.get_reps(y)
                    surrogate = torch.dot(reps.flatten(), gradient.flatten())
                    if sync_flag:
                        model.require_backward_grad_sync = True
                    if self.fp16:  # [added]
                        self.scaler._enabled = False
                        self.backward_fn(surrogate)
                        self.scaler._enabled = True
                    else:
                        self.backward_fn(surrogate)  # [modified]
        else:  # [use base model (i.e. SimpleLitModel)]

            # [remove no_sync_except_last: pytorch lightning would handle gradient sync automatically]
            for x, state, gradient in zip(
                model_inputs, random_states, cached_gradients
            ):
                with state:
                    y = self.model_call(model, x)
                reps = self.get_reps(y)
                surrogate = torch.dot(reps.flatten(), gradient.flatten())
                if self.fp16:  # [added]
                    self.scaler._enabled = False
                    self.backward_fn(surrogate)
                    self.scaler._enabled = True
                else:
                    self.backward_fn(surrogate)  # [added]

    def cache_step(
        self, *model_inputs, no_sync_except_last: bool = False, **loss_kwargs
    ) -> Tuple[Tensor, Tensor]:
        all_reps = []
        all_rnd_states = []

        model_inputs = [
            self.split_inputs(x, chunk_size)
            for x, chunk_size in zip(model_inputs, self.chunk_sizes)
        ]

        for model, x in zip(self.models, model_inputs):
            model_reps, rnd_states = self.forward_no_grad(model, x)
            all_reps.append(model_reps)
            all_rnd_states.append(rnd_states)

        cache, loss = self.build_cache(*all_reps, **loss_kwargs)
        cache = [c.split(chunk_size) for c, chunk_size in zip(cache, self.chunk_sizes)]

        for model, x, model_cache, rnd_states in zip(
            self.models, model_inputs, cache, all_rnd_states
        ):
            self.forward_backward(
                model,
                x,
                model_cache,
                rnd_states,
                no_sync_except_last=no_sync_except_last,
            )

        return loss

In [2]:
class LearnableInfoNCELossText(nn.Module):
    def __init__(self):
        super(LearnableInfoNCELossText, self).__init__()
        self.w = nn.Parameter(torch.tensor(float(torch.log(torch.tensor(10.0))), requires_grad=True))
        self.b = nn.Parameter(torch.tensor(-10.0, requires_grad=True))

    def forward(self, emb, mask, indices):
        # Calculate the similarity matrix
        similarity_matrix = torch.matmul(emb, emb.T)

        # Apply scaling and bias to the similarity matrix
        logits = torch.exp(self.w) * similarity_matrix + self.b
        logits = logits.masked_fill(~mask, float('-inf'))

        # Apply log softmax to logits
        # log_probs = torch.log(logits.softmax(1))
        log_probs = F.log_softmax(logits, dim=1)

        losses = []
        for idx, indices_row in enumerate(indices):
            log_probs_row = log_probs[idx, indices_row]
            log_probs_row_mask = torch.isfinite(log_probs_row)
            losses.append(log_probs_row[log_probs_row_mask])
        loss = -torch.mean(torch.cat(losses))
        if torch.isnan(loss):
            raise ValueError()
        return loss

class LearnableInfoNCELossMM(nn.Module):
    def __init__(self):
        super(LearnableInfoNCELossMM, self).__init__()
        self.w = nn.Parameter(torch.tensor(float(torch.log(torch.tensor(10.0))), requires_grad=True))
        self.b = nn.Parameter(torch.tensor(-10.0, requires_grad=True))

    def forward(self, image_emb, text_emb, indices):
        # Calculate the similarity matrix
        similarity_matrix = torch.matmul(text_emb, image_emb.T)

        # Apply scaling and bias to the similarity matrix
        logits = torch.exp(self.w) * similarity_matrix + self.b
        log_probs = F.log_softmax(logits, dim=1)

        rows, cols = zip(*indices)
        loss = -log_probs[torch.tensor(rows), torch.tensor(cols)]
        loss = torch.sum(loss)/text_emb.shape[0]
        if torch.isnan(loss):
            raise ValueError()
        return loss

In [3]:
import io
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
import bm25s
import Stemmer

from tqdm.auto import tqdm
from PIL import Image
from glob import glob
from datasets import load_dataset, concatenate_datasets, load_from_disk, get_dataset_config_names, interleave_datasets, get_dataset_config_info
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import ndcg_score, recall_score
# from bitsandbytes.optim import AdamW8b

In [4]:
SEED = 42
BATCH_SIZE = 1024
MINI_BATCH_SIZE = 32
BM25_NEG_THRESHOLD = 2.5
MIN_TEXT_LENGTH = 20

TEXT_MODEL_NAME = "Alibaba-NLP/gte-base-en-v1.5"
IMAGE_MODEL_NAME = "microsoft/dit-base"

In [5]:
L.seed_everything(SEED)
torch.set_float32_matmul_precision('medium')

Seed set to 42


In [6]:
train_datasets = []
for dataset_name in get_dataset_config_names('jwengr/document-vqa-finetune'):
    train_datasets.append(load_dataset('jwengr/document-vqa-finetune', name=dataset_name, split='train'))
train_dataset = concatenate_datasets(train_datasets)
train_dataset = train_dataset.shuffle(seed=SEED)

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/62 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/297 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/289 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/140 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/136 [00:00<?, ?it/s]

In [7]:
eval_datasets = []
for dataset_name in get_dataset_config_names('jwengr/document-vqa-evaluation'):
    eval_datasets.append(load_dataset('jwengr/document-vqa-evaluation', name=dataset_name, split='train'))
eval_dataset = concatenate_datasets(eval_datasets)

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

In [8]:
class TrainCollateFn:
    def __init__(self, text_model_name, image_model_name, bm25_neg_threshold=BM25_NEG_THRESHOLD):
        self.text_tokenizer = AutoTokenizer.from_pretrained(text_model_name)
        self.image_processor = AutoImageProcessor.from_pretrained(image_model_name)
        self.retriever = bm25s.BM25(backend="numba")
        self.retriever.activate_numba_scorer()
        self.stemmer = Stemmer.Stemmer("english")
        self.bm25_neg_threshold = bm25_neg_threshold
        
    
    def __call__(self, examples):
        images = []
        questions = []
        text_indices = []
        mm_indices = []
        for example in examples:
            text_labels_row = list(range(len(text_indices), len(text_indices)+len(example['questions'])))
            for _ in range(len(example['questions'])):
                text_indices.append(text_labels_row)
            mm_labels_row = [(idx + len(questions) , label+len(images))for idx, label in enumerate(example['labels'])]
            mm_indices.extend(mm_labels_row)
            images_row = [Image.open(io.BytesIO(image_bytes)) for image_bytes in example['image_bytes']]
            images.extend(images_row)
            questions.extend(example['questions'])

        corpus_tokens = bm25s.tokenize(questions, stopwords="en", stemmer=self.stemmer, show_progress=False, leave=True)
        self.retriever.index(corpus_tokens, show_progress=False, leave_progress=True)
        scores = []
        for idx, corpus_ids in enumerate(corpus_tokens.ids):
            tokenized = bm25s.tokenization.Tokenized(ids=[corpus_ids], vocab=corpus_tokens.vocab)
            result, score = self.retriever.retrieve(tokenized, k=len(questions))               
            result, score = result[0], score[0]
            score_row = score[np.argsort(result)]
            scores.append(score_row)
        scores = np.vstack(scores)
        mask = scores<self.bm25_neg_threshold
        mask[np.arange(len(mask)), np.arange(len(mask))] = False
        mask = torch.from_numpy(mask)
        
        image_inputs = self.image_processor(images=images, return_tensors='pt')
        text_inputs = self.text_tokenizer(text=questions,  max_length=8192, padding=True, truncation=True, return_tensors='pt')

        return {
            'inputs':{
                'image_inputs': dict(image_inputs),  # Image inputs remain on CPU unless moved explicitly
                'text_inputs': dict(text_inputs),     # Anchor embeddings are computed on the GPU if device is 'cuda'
            },
            'mm_indices':mm_indices,
            'text_indices':text_indices,
            'text_mask':mask
        }


In [9]:
class EvalCollateFn:
    def __init__(self, text_model_name, image_model_name):
        self.text_tokenizer = AutoTokenizer.from_pretrained(text_model_name)
        self.image_processor = AutoImageProcessor.from_pretrained(image_model_name)

    def __call__(self, examples):
        example = examples[0]
        images = [Image.open(io.BytesIO(image_byte)) for image_byte in example['image_bytes']]
        image_inputs = self.image_processor(images=images, return_tensors='pt')
        text_inputs = [self.text_tokenizer(text=question,  max_length=8192, padding=True, truncation=True, return_tensors='pt') for question in example['questions']] 
       
        return {
            'dataset': example['dataset'],
            'doc_types':example['doc_types'],
            'image_inputs':image_inputs,
            'text_inputs': text_inputs,
            'labels': example['labels'],
            'label_types': example['label_types']
        }

    

In [10]:
train_dataloader = DataLoader(train_dataset, collate_fn=TrainCollateFn(text_model_name=TEXT_MODEL_NAME, image_model_name=IMAGE_MODEL_NAME), shuffle=True, batch_size=BATCH_SIZE, drop_last=True)
eval_dataloader = DataLoader(eval_dataset, collate_fn=EvalCollateFn(text_model_name=TEXT_MODEL_NAME, image_model_name=IMAGE_MODEL_NAME), batch_size=1)

In [11]:
class LitDocEmbModel(L.LightningModule):
    def __init__(
        self, 
        text_model_name,
        image_model_name,
        mini_batch_size=32
    ):
        super().__init__()
        self.save_hyperparameters()
        self.text_model = AutoModel.from_pretrained(text_model_name, trust_remote_code=True)
        self.text_model.train()
        self.image_model = AutoModel.from_pretrained(image_model_name)
        self.image_model.train()
        self.loss_text = LearnableInfoNCELossText()
        self.loss_mm = LearnableInfoNCELossMM()
        self.mini_batch_size = mini_batch_size
        self.evaluation_df = pd.DataFrame(columns=['dataset', 'probs', 'labels', 'doc_type', 'label_types'])
        self.strict_loading = False
        self.automatic_optimization = False

    def init_grad_cache(self, scaler, ddp_module):
        self.trainer.strategy.precision_plugin.forward_context = nullcontext
        self.grad_cache_text = PLGradCache(
            models=[ddp_module],
            chunk_sizes=self.mini_batch_size,
            loss_fn=self.calculate_loss_text,
            fp16=True,
            scaler=scaler, # needed when using automatic_optimization is off and fp16 is on
            backward_fn=self.manual_backward, # needed when automatic_optimization is off
        )
        self.grad_cache_mm = PLGradCache(
            models=[ddp_module],
            chunk_sizes=self.mini_batch_size,
            loss_fn=self.calculate_loss_mm,
            fp16=True,
            scaler=scaler, # needed when using automatic_optimization is off and fp16 is on
            backward_fn=self.manual_backward, # needed when automatic_optimization is off
        )
        self.train_text=True

    def calculate_loss_text(self, text_embeddings, text_mask, text_indices):
        return self.loss_text(text_embeddings, text_mask, text_indices)

    def calculate_loss_mm(self, image_embeddings, text_embeddings, mm_indices):
        return self.loss_mm(image_embeddings, text_embeddings, mm_indices)

    def get_image_embeds(self, pixel_values):
        pooler_output = self.image_model(pixel_values).pooler_output
        image_embeds = F.normalize(pooler_output, p=2, dim=-1)
        return image_embeds

    def get_text_embeds(self, input_ids, attention_mask, token_type_ids):
        text_embeds = self.text_model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids).last_hidden_state[:,0]
        text_embeds = F.normalize(text_embeds, p=2, dim=-1)
        return text_embeds

    def freeze_image_model(self, requires_grad=False):
        for param in self.image_model.parameters():
            param.requires_grad = requires_grad
    
    def freeze_text_model(self, requires_grad):
        for param in self.text_model.parameters():
            param.requires_grad = requires_grad
    
    def forward(self, input_ids=None, attention_mask=None, pixel_values=None, token_type_ids=None):
        if self.train_text:
            text_embeds = self.get_text_embeds(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
            return text_embeds
        else :
            image_embeds = self.get_image_embeds(pixel_values=pixel_values)
            return image_embeds

    def on_train_start(self): # initialize grad cache here
        self.init_grad_cache(self.trainer.scaler, self.trainer.strategy.model)

    def on_train_epoch_end(self):
        torch.cuda.empty_cache()
        gc.collect()

    def configure_optimizers(self):
        # from torch.optim import AdamW
        from bitsandbytes.optim import AdamW8bit
        opt = AdamW8bit(self.parameters(), lr=1e-5)
        return opt
        
    def training_step(self, batch, batch_idx):
        optimizer = self.optimizers()
        inputs, mm_indices, text_indices, text_mask = batch['inputs'], batch['mm_indices'], batch['text_indices'], batch['text_mask']
        
        #train text
        self.train_text = True
        self.freeze_text_model(requires_grad=True)
        self.freeze_image_model(requires_grad=False)
        optimizer.zero_grad()
        loss_text = self.grad_cache_text(
            inputs['text_inputs'],
            no_sync_except_last=False,
            text_mask=text_mask,
            text_indices=text_indices
        )
        optimizer.step()
        self.log(f"train_loss_text", loss_text, on_step=True, on_epoch=False)
        torch.cuda.empty_cache()
        gc.collect()
        #train mm
        self.train_text = False
        self.freeze_text_model(requires_grad=False)
        self.freeze_image_model(requires_grad=True)
        
        num_samples = len(inputs['text_inputs']['input_ids'])
        text_embeddings = []
        with torch.no_grad():
            # Iterate over mini-batches
            for i in range(0, num_samples, self.mini_batch_size):
                # Get the current batch of inputs
                mini_batch_text_inputs = {key: val[i:i+self.mini_batch_size] for key, val in inputs['text_inputs'].items()}
                mini_batch_text_embeddings = self.get_text_embeds(**mini_batch_text_inputs)
                text_embeddings.append(mini_batch_text_embeddings)
            text_embeddings = torch.cat(text_embeddings, dim=0)
        
        optimizer.zero_grad()
        loss_mm = self.grad_cache_mm(
            inputs['image_inputs'],
            no_sync_except_last=False,
            text_embeddings=text_embeddings,
            mm_indices=mm_indices
        )
        optimizer.step()
        self.log(f"train_loss_mm", loss_mm, on_step=True, on_epoch=False)
        torch.cuda.empty_cache()
        gc.collect()


        return 
        
    def validation_step(self, batch):
        image_inputs, text_inputs = batch['image_inputs'], batch['text_inputs']
        image_embeds = self.get_image_embeds(**image_inputs).detach().cpu().numpy()

        # Process each question embedding and record metrics
        for text_inputs, labels, doc_type, label_types in zip(text_inputs, batch['labels'], batch['doc_types'], batch['label_types']):
            df_index = len(self.evaluation_df)
            text_embeds = self.get_text_embeds(**text_inputs).detach().cpu().numpy()
            probs = np.dot(text_embeds, image_embeds.T)[0]
            self.evaluation_df.loc[df_index, 'dataset'] = batch['dataset']
            self.evaluation_df.loc[df_index, 'probs'] = probs
            self.evaluation_df.loc[df_index, 'labels'] = labels
            self.evaluation_df.loc[df_index, 'doc_type'] = doc_type
            self.evaluation_df.loc[df_index, 'label_types'] = label_types
        return

    def on_validation_epoch_end(self):
        # Initialize metrics storage
        dataset_metrics = {}
        total_recall1 = []
        total_ndcg = []

        # Iterate over unique datasets
        for dataset in self.evaluation_df['dataset'].unique():
            df_filtered = self.evaluation_df[self.evaluation_df['dataset'] == dataset]

            # Initialize metric lists
            single_recall1_list = []
            single_recall3_list = []
            single_recall5_list = []
            multi_ndcg_list = []

            # Iterate over each row (sample)
            for index, row in df_filtered.iterrows():
                probs = row['probs']   # Model predictions (probabilities)
                labels = row['labels']  # Ground truth labels

                # Handle single-label case (len(labels) == 1)
                if len(labels) == 1:
                    # Calculate Recall@1, Recall@3, and Recall@5
                    top_indices = np.argsort(-probs)[:5]  # Get top 5 predictions
                    single_recall1 = int(labels[0] == top_indices[0])  # Recall@1
                    single_recall3 = int(labels[0] in top_indices[:3])  # Recall@3
                    single_recall5 = int(labels[0] in top_indices[:5])  # Recall@5

                    single_recall1_list.append(single_recall1)
                    single_recall3_list.append(single_recall3)
                    single_recall5_list.append(single_recall5)
                    total_recall1.append(single_recall1)  # Accumulate for total Recall@1

                # Handle multi-label case (len(labels) > 1)
                else:
                    # Create relevance vector for NDCG
                    relevance = np.zeros(len(probs))
                    relevance[labels] = 1  # Assume relevance of 1 for relevant items

                    # Calculate NDCG
                    ndcg = ndcg_score([relevance], [probs])
                    multi_ndcg_list.append(ndcg)
                    total_ndcg.append(ndcg)  # Accumulate for total NDCG

            # Store dataset-level metrics
            dataset_metrics[dataset] = {
                'recall@1': np.mean(single_recall1_list) if single_recall1_list else None,
                'recall@3': np.mean(single_recall3_list) if single_recall3_list else None,
                'recall@5': np.mean(single_recall5_list) if single_recall5_list else None,
                'NDCG': np.mean(multi_ndcg_list) if multi_ndcg_list else None
            }

            # Log dataset-specific metrics
            self.log(f'{dataset} Recall@1 (single-label)', dataset_metrics[dataset]['recall@1'])
            self.log(f'{dataset} Recall@3 (single-label)', dataset_metrics[dataset]['recall@3'])
            self.log(f'{dataset} Recall@5 (single-label)', dataset_metrics[dataset]['recall@5'])
            if dataset_metrics[dataset]['NDCG'] is not None:
                self.log(f'{dataset} NDCG (multi-label)', dataset_metrics[dataset]['NDCG'])

        # Compute total metrics across all datasets
        total_recall1_mean = np.mean(total_recall1) if total_recall1 else None
        total_ndcg_mean = np.mean(total_ndcg) if total_ndcg else None

        # Log total metrics
        self.log('recall_at_1', total_recall1_mean)
        self.log('ndcg', total_ndcg_mean)

        # Reset the evaluation DataFrame for the next epoch
        self.evaluation_df = pd.DataFrame(columns=['dataset', 'probs', 'labels', 'doc_type', 'label_types'])
        torch.cuda.empty_cache()
        gc.collect()
        return


In [12]:
lit_model = LitDocEmbModel.load_from_checkpoint(
    './checkpoint/gte-base-en-v1.5-dit-base-finetune-batch=1024-epoch=3-recall_at_1=0.3140.ckpt',
    text_model_name=TEXT_MODEL_NAME, image_model_name=IMAGE_MODEL_NAME, mini_batch_size=MINI_BATCH_SIZE
)

Some weights of BeitModel were not initialized from the model checkpoint at microsoft/dit-base and are newly initialized: ['beit.pooler.layernorm.bias', 'beit.pooler.layernorm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
# lit_model = LitDocEmbModel(text_model_name=TEXT_MODEL_NAME, image_model_name=IMAGE_MODEL_NAME, mini_batch_size=MINI_BATCH_SIZE)

In [14]:
checkpoint_callback = ModelCheckpoint(
    monitor='recall_at_1',
    verbose=True,
    save_top_k=1,
    mode='max',
    dirpath='checkpoint',
    filename=f"{TEXT_MODEL_NAME.split('/')[-1]}-{IMAGE_MODEL_NAME.split('/')[-1]}-finetune-batch={BATCH_SIZE}"+"-{epoch:2d}-{recall_at_1:.4f}"
)

earlystop_callback = EarlyStopping(
    monitor="recall_at_1", 
    patience=6, 
    verbose=True,
    mode="max"
)

In [15]:
trainer = L.Trainer(
    max_epochs=1000, 
    precision=16, 
    callbacks=[checkpoint_callback, earlystop_callback],
    val_check_interval=0.5,
    num_sanity_val_steps=0
)

/usr/local/lib/python3.11/dist-packages/lightning/fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [ ]:
trainer.fit(lit_model, train_dataloader, eval_dataloader)

/usr/local/lib/python3.11/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /workspace/checkpoint exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type                     | Params | Mode 
-----------------------------------------------------------------
0 | text_model  | NewModel                 | 136 M  | train
1 | image_model | BeitModel                | 85.8 M | train
2 | loss_text   | LearnableInfoNCELossText | 2      | train
3 | loss_mm     | LearnableInfoNCELossMM   | 2      | train
-----------------------------------------------------------------
222 M     Trainable params
0         Non-trainable params
222 M     Total params
890.339   Total estimated model params size (MB)
404       Modules in train mode
0         Modules in eval mode
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottle

Training: |          | 0/? [00:00<?, ?it/s]

In [15]:
trainer.validate(lit_model, eval_dataloader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\dust\anaconda3\envs\py311_torch2\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Validation: |                                                                                    | 0/? [00:00<…

C:\Users\dust\.cache\huggingface\modules\transformers_modules\Alibaba-NLP\new-impl\40ced75c3017eb27626c9d4ea981bde21a2662f4\modeling.py:579: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃            Validate metric            ┃             DataLoader 0              ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   mm_long_bench NDCG (multi-label)    │          0.4033726155757904           │
│ mm_long_bench Recall@1 (single-label) │          0.04615384712815285          │
│ mm_long_bench Recall@3 (single-label) │          0.10769230872392654          │
│ mm_long_bench Recall@5 (single-label) │          0.1538461595773697           │
│   mp_docvqa Recall@1 (single-label)   │          0.22170811891555786          │
│   mp_docvqa Recall@3 (single-label)   │          0.42259493470191956          │
│   mp_docvqa Recall@5 (single-label)   │          0.5263158082962036           │
│                 ndcg                  │          0.4033726155757904           │
│              recall_at_1              │          0.21953541040420532          │
└───────────────────────────────────────┴───────────────────────────────────────┘

[{'mm_long_bench Recall@1 (single-label)': 0.04615384712815285,
  'mm_long_bench Recall@3 (single-label)': 0.10769230872392654,
  'mm_long_bench Recall@5 (single-label)': 0.1538461595773697,
  'mm_long_bench NDCG (multi-label)': 0.4033726155757904,
  'mp_docvqa Recall@1 (single-label)': 0.22170811891555786,
  'mp_docvqa Recall@3 (single-label)': 0.42259493470191956,
  'mp_docvqa Recall@5 (single-label)': 0.5263158082962036,
  'recall_at_1': 0.21953541040420532,
  'ndcg': 0.4033726155757904}]